## RAG 评估指标 · 04 — 完整诊断流水线

**本 Notebook 目标：** 整合 Context Recall、Context Precision、Faithfulness、Answer Relevancy 四项指标，自动生成诊断报告和优先级建议。

## 学习目标

- 将四项核心指标串联为端到端诊断流水线
- 根据阈值对比生成可操作的优化建议
- 按缺口大小排序优先级，输出 HTML 诊断报告

```bash
pip install openai python-dotenv
```

> 分数低了，下一步该动哪里？

## RAG 评估指标 · 04 — 完整诊断流水线

**本 Notebook 目标：** 整合 Context Recall、Context Precision、Faithfulness、Answer Relevancy 四项指标，自动生成诊断报告和优先级建议。

## 学习目标

- 将四项核心指标串联为端到端诊断流水线
- 根据阈值对比生成可操作的优化建议
- 按缺口大小排序优先级，输出 HTML 诊断报告

```bash
pip install openai python-dotenv
```

> 分数低了，下一步该动哪里？

## 端到端诊断流水线

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    SAMPLE[RAG 样本] --> M1[Context Recall]
    SAMPLE --> M2[Context Precision]
    SAMPLE --> M3[Faithfulness]
    SAMPLE --> M4[Answer Relevancy]
    M1 --> DIAG[generate_diagnostics]
    M2 --> DIAG
    M3 --> DIAG
    M4 --> DIAG
    DIAG --> PRI[priority_rank 按缺口排序]
    PRI --> OUT[终端报告 + HTML]
```

---
# Chapter 6 · 诊断流程

## 四象限诊断框架

| 象限 | Precision | Recall | 首要行动 |
|------|-----------|--------|----------|
| 右上 | 高 | 高 | 检查 Faithfulness + Relevancy |
| 左上 | 高 | 低 | 调 chunk_size 或换 embedding |
| 右下 | 低 | 高 | 加 reranker 或降低 Top-K |
| 左下 | 低 | 低 | 优先修 embedding + chunking |

## Precision × Recall 四象限

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 40}}}%%
flowchart TB
    subgraph HIGH_R["Recall 高"]
        Q1[Precision 高 → 查 Generator]
        Q2[Precision 低 → 加 Reranker]
    end
    subgraph LOW_R["Recall 低"]
        Q3[Precision 高 → 调 chunk/embedding]
        Q4[双低 → 最危险 先修 Retriever]
    end
```

## 诊断决策树

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    START[评估完成 找最低指标] --> R{Recall 低?}
    START --> P{Precision 低?}
    START --> F{Faithfulness 低?}
    START --> A{Relevancy 低?}
    R -->|是| R1[增大 top_k / 调 chunk]
    P -->|是| P1[Reranker / 减小 K]
    F -->|是| F1[严格 Prompt / 降 temperature]
    A -->|是| A1[Query 扩展 / HyDE]
```

## 分项修复策略

### Recall 低 → 调 Chunking

| 策略 | chunk_size | overlap | Recall 影响 |
|------|-----------|---------|------------|
| 小块密集 | 128-256 | 20-50 | 高 |
| 中块平衡 | 512 | 50-100 | 中（默认） |
| 大块完整 | 1024+ | 100-200 | 低 |

> **Recall 低 → 先减小 chunk_size + 增加 overlap，再换 embedding**

### Precision 低 → Top-K 与 Reranker

- K 增大 → Recall ↑、Precision ↓
- K 减小 → Precision ↑、Recall ↓
- **两阶段**：Bi-Encoder Top-20 → Cross-Encoder Top-5

### Faithfulness 低 → Prompt

- 添加「只能使用参考资料」约束
- 要求引用上下文
- 降低 temperature

### Relevancy 低 → Query 扩展

- **HyDE**：假设答案 → 向量检索
- **多查询**：变体 query 合并去重

## Retriever vs Generator 分工

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart LR
    subgraph RET[Retriever 指标]
        CR[Context Recall]
        CP[Context Precision]
    end
    subgraph GEN[Generator 指标]
        FH[Faithfulness]
        AR[Answer Relevancy]
    end
    RET --> FIX1[chunk / embedding / reranker]
    GEN --> FIX2[Prompt / query rewrite]
```

---
# Part 1 · 环境准备（代码实战）

从 `04_RAG_Evaluation/.env` 加载 `OPENAI_API_KEY`。

| 模式 | 条件 | Faithfulness |
|------|------|--------------|
| **Mock（默认）** | 无 Key 或 `USE_MOCK=True` | 使用预设分数 0.67，触发诊断建议 |
| **Live** | `.env` 中有 Key | 调用 DeepSeek LLM-as-Judge |

> 可直接 **Run All**，无需 API Key。

In [ ]:
import os
import json
from html import escape
from pathlib import Path

try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv", "openai", "-q"])
    from dotenv import load_dotenv

from openai import OpenAI

# 无 API Key 时设为 True，Faithfulness 使用 MOCK 分数，可 Run All
USE_MOCK = False

for _candidate in [Path("..") / ".env", Path("../..") / ".env"]:
    if _candidate.exists():
        load_dotenv(_candidate)
        break

api_key = os.getenv("OPENAI_API_KEY", "").strip()
if USE_MOCK or not api_key:
    client = None
    print("⚠️  未配置 API Key → Faithfulness 使用 Mock 分数（USE_MOCK=True 或填写 .env）")
else:
    client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
    print("✅ DeepSeek 客户端已初始化")

---
# Part 2 · 阈值配置与指标函数

## 四项指标计算来源

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    IDS[retrieved_ids + relevant_ids] --> CR[compute_recall]
    IDS --> CP[compute_precision]
    AC[answer + context] --> FH[_faithfulness_llm]
    QA[query + answer] --> AR[_answer_relevancy_keyword]
```

In [ ]:
THRESHOLDS = {
    "context_recall":    0.7,
    "context_precision": 0.6,
    "faithfulness":      0.8,
    "answer_relevancy":  0.7,
}


def compute_precision(retrieved_ids: list, relevant_ids: set) -> float:
    """Context Precision = 相关检索数 / 检索总数"""
    if not retrieved_ids:
        return 0.0
    return len([d for d in retrieved_ids if d in relevant_ids]) / len(retrieved_ids)


def compute_recall(retrieved_ids: list, relevant_ids: set) -> float:
    """Context Recall = 相关检索数 / 全部相关数"""
    if not relevant_ids:
        return 0.0
    return len([d for d in retrieved_ids if d in relevant_ids]) / len(relevant_ids)


def _faithfulness_llm(answer: str, context: str, llm_client: OpenAI) -> float:
    """使用 LLM 计算忠实度分数（原子声明验证方法）"""
    extract_prompt = (
        "请将以下答案拆解为独立的原子性声明，每条仅包含一个事实。\n\n"
        f"答案：\n{answer}\n\n"
        "以 JSON 数组返回，不含任何其他文字：\n"
        '["声明1", "声明2"]'
    )
    resp = llm_client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": extract_prompt}],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        claims = json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("["), raw.rfind("]") + 1
        claims = json.loads(raw[start:end]) if start != -1 and end > start else [raw]

    if not claims:
        return 0.0

    supported = 0
    for claim in claims:
        verify_prompt = (
            "请判断以下声明是否可从给定上下文中得到支持。\n\n"
            f"上下文：\n{context}\n\n"
            f"声明：\n{claim}\n\n"
            '{"supported": true/false}，不含其他文字。'
        )
        vresp = llm_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": verify_prompt}],
            temperature=0,
        )
        vraw = vresp.choices[0].message.content.strip()
        try:
            verdict = json.loads(vraw)
        except json.JSONDecodeError:
            s, e = vraw.find("{"), vraw.rfind("}") + 1
            try:
                verdict = json.loads(vraw[s:e]) if s != -1 and e > s else {}
            except json.JSONDecodeError:
                verdict = {}
        if verdict.get("supported", False):
            supported += 1

    return supported / len(claims)


def _tokenize(text: str) -> set:
    """字符级分词，过滤标点"""
    stop = set("，。！？、；：\"\"''（）【】《》 \n\t,.!?;:()'\"[]<>")
    return set(ch for ch in text if ch not in stop)


def _answer_relevancy_keyword(question: str, answer: str) -> float:
    """基于 Jaccard 相似度的答案相关性"""
    q_set = _tokenize(question)
    a_set = _tokenize(answer)
    if not q_set or not a_set:
        return 0.0
    return len(q_set & a_set) / len(q_set | a_set)

---
# Part 3 · 模拟 RAG 样本数据

In [ ]:
RAG_SAMPLE = {
    "query": "如何通过 RAG 技术减少大语言模型的幻觉？",
    "retrieved_ids": ["doc1", "doc3", "doc5"],
    "relevant_ids": {"doc1", "doc5", "doc6", "doc7"},
    "context": (
        "RAG 技术通过在生成前检索相关文档，将外部知识注入模型上下文，减少幻觉。"
        "LLM 幻觉是指模型生成与上下文不一致或凭空捏造的内容。"
    ),
    "answer": (
        "RAG 通过检索相关文档并注入上下文，引导模型基于事实回答，"
        "从而有效减少幻觉现象。此外，RAG 还能提升回答的时效性。"
    ),
}

MOCK_FAITHFULNESS = 0.67  # 故意设低，触发诊断建议

---
# Part 4 · 诊断引擎

## 诊断引擎流程

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    MET[metrics dict] --> GD[generate_diagnostics]
    TH[THRESHOLDS] --> GD
    GD --> PASS{score >= threshold?}
    PASS -->|否| REC[recommendation]
    PASS -->|是| OK[通过]
    REC --> PR[priority_rank 按缺口排序]
    PR --> FC[failure_category 总结]
    FC --> HTML[write_html_report]
```

In [ ]:
def generate_diagnostics(metrics_dict: dict) -> list:
    """根据各项指标与阈值的对比，生成可操作的优化建议列表"""
    diagnostics = []
    checks = [
        ("context_recall", "Context Recall（上下文召回率）", "建议增大 top_k 或减小 chunk_size，确保相关文档被检索到"),
        ("context_precision", "Context Precision（上下文精确率）", "建议添加 Reranker 或减小 top_k，过滤掉不相关的检索结果"),
        ("faithfulness", "Faithfulness（忠实度）", "建议优化 System Prompt，添加 '仅基于以下内容回答' 约束，减少模型幻觉"),
        ("answer_relevancy", "Answer Relevancy（答案相关性）", "建议优化 Query，使用 Query Expansion 或 HyDE 提升问题表达质量"),
    ]
    for key, label, recommendation in checks:
        score = metrics_dict.get(key, 0.0)
        threshold = THRESHOLDS[key]
        passed = score >= threshold
        diagnostics.append({
            "metric": label,
            "key": key,
            "score": score,
            "threshold": threshold,
            "pass": passed,
            "recommendation": recommendation if not passed else None,
        })
    return diagnostics


def priority_rank(diagnostics: list) -> list:
    """对失败的指标按缺口大小（threshold - score）排序"""
    failed = [d for d in diagnostics if not d["pass"]]
    failed.sort(key=lambda d: d["threshold"] - d["score"], reverse=True)
    return failed


def render_score_bar(score: float) -> str:
    bar_filled = int(score * 10)
    return "█" * bar_filled + "░" * (10 - bar_filled)


def failure_category(metrics_dict: dict) -> str:
    low_recall = metrics_dict["context_recall"] < THRESHOLDS["context_recall"]
    low_precision = metrics_dict["context_precision"] < THRESHOLDS["context_precision"]
    low_faithfulness = metrics_dict["faithfulness"] < THRESHOLDS["faithfulness"]
    low_relevancy = metrics_dict["answer_relevancy"] < THRESHOLDS["answer_relevancy"]

    if low_recall and low_precision:
        return "Retriever 召回不足且噪声偏高：优先检查 chunking、embedding、top_k 与 reranker。"
    if low_recall:
        return "Retriever 漏召回：生成器可能拿不到回答问题所需的关键上下文。"
    if low_precision:
        return "Retriever 噪声偏高：生成器容易被不相关上下文干扰。"
    if low_faithfulness:
        return "Generator 幻觉风险：答案包含上下文无法支持的内容。"
    if low_relevancy:
        return "Generator 没有正面回答问题：需要改进 query rewrite 或 question-aware prompt。"
    return "整体健康：当前样本的四项核心指标均达标。"


def write_html_report(sample, metrics_dict, diagnostics_list, failed_list, output_path: Path) -> None:
    """输出一个可离线打开的诊断 HTML 报告"""
    rows = []
    for item in diagnostics_list:
        status_text = "通过" if item["pass"] else "未达标"
        status_class = "pass" if item["pass"] else "fail"
        recommendation = item["recommendation"] or "保持当前配置，继续扩大评估集验证稳定性。"
        rows.append(
            "<tr>"
            f"<td>{escape(item['metric'])}</td>"
            f"<td>{item['score']:.2f}</td>"
            f"<td>{item['threshold']:.2f}</td>"
            f"<td><span class=\"pill {status_class}\">{status_text}</span></td>"
            f"<td>{escape(recommendation)}</td>"
            "</tr>"
        )

    priority_items = []
    if failed_list:
        for index, item in enumerate(failed_list, start=1):
            gap = item["threshold"] - item["score"]
            priority_items.append(
                "<li>"
                f"<strong>优先级 {index}: {escape(item['metric'])}</strong>"
                f"<span>当前 {item['score']:.2f} / 阈值 {item['threshold']:.2f} / 缺口 {gap:.2f}</span>"
                f"<p>{escape(item['recommendation'])}</p>"
                "</li>"
            )
    else:
        priority_items.append("<li><strong>所有指标达标</strong><span>暂无紧急修复项。</span></li>")

    cards = []
    for key, label in [
        ("context_recall", "Context Recall"),
        ("context_precision", "Context Precision"),
        ("faithfulness", "Faithfulness"),
        ("answer_relevancy", "Answer Relevancy"),
    ]:
        score = metrics_dict[key]
        cards.append(
            "<article class=\"metric-card\">"
            f"<span>{label}</span>"
            f"<strong>{score:.2f}</strong>"
            f"<div class=\"meter\"><i style=\"width: {score * 100:.0f}%\"></i></div>"
            f"<small>阈值 {THRESHOLDS[key]:.2f}</small>"
            "</article>"
        )

    html = f"""<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>RAG Diagnostic Report</title>
  <style>
    :root {{
      color-scheme: light;
      --ink: #172026;
      --muted: #667085;
      --line: #d0d7de;
      --bg: #f6f8fa;
      --panel: #ffffff;
      --green: #137333;
      --red: #b42318;
      --blue: #1f6feb;
    }}
    * {{ box-sizing: border-box; }}
    body {{
      margin: 0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: var(--ink);
      background: var(--bg);
    }}
    main {{
      max-width: 1120px;
      margin: 0 auto;
      padding: 32px 20px 48px;
    }}
    h1 {{ margin: 0 0 8px; font-size: 32px; }}
    h2 {{ margin: 32px 0 12px; font-size: 20px; }}
    p {{ color: var(--muted); line-height: 1.6; }}
    .summary {{
      padding: 18px 20px;
      border: 1px solid var(--line);
      border-left: 5px solid var(--blue);
      background: var(--panel);
      border-radius: 8px;
    }}
    .grid {{
      display: grid;
      grid-template-columns: repeat(auto-fit, minmax(210px, 1fr));
      gap: 12px;
      margin-top: 20px;
    }}
    .metric-card {{
      background: var(--panel);
      border: 1px solid var(--line);
      border-radius: 8px;
      padding: 16px;
    }}
    .metric-card span, .metric-card small {{ color: var(--muted); }}
    .metric-card strong {{ display: block; margin: 8px 0; font-size: 34px; }}
    .meter {{
      height: 8px;
      overflow: hidden;
      background: #eaeef2;
      border-radius: 99px;
      margin: 10px 0 8px;
    }}
    .meter i {{ display: block; height: 100%; background: var(--blue); }}
    table {{
      width: 100%;
      border-collapse: collapse;
      background: var(--panel);
      border: 1px solid var(--line);
      border-radius: 8px;
      overflow: hidden;
    }}
    th, td {{
      padding: 12px 14px;
      text-align: left;
      border-bottom: 1px solid var(--line);
      vertical-align: top;
    }}
    th {{ background: #eef2f6; }}
    tr:last-child td {{ border-bottom: 0; }}
    .pill {{
      display: inline-block;
      min-width: 58px;
      padding: 3px 8px;
      border-radius: 999px;
      font-size: 13px;
      text-align: center;
      font-weight: 700;
    }}
    .pill.pass {{ color: var(--green); background: #e6f4ea; }}
    .pill.fail {{ color: var(--red); background: #fce8e6; }}
    ol {{
      display: grid;
      gap: 10px;
      padding-left: 24px;
    }}
    li {{
      background: var(--panel);
      border: 1px solid var(--line);
      border-radius: 8px;
      padding: 12px 14px;
    }}
    li span {{ display: block; margin: 5px 0; color: var(--muted); }}
    code {{
      background: #eef2f6;
      border-radius: 4px;
      padding: 2px 5px;
    }}
  </style>
</head>
<body>
  <main>
    <h1>RAG Diagnostic Report</h1>
    <p>自动评估 Retriever 与 Generator 的核心指标，并输出优先修复路径。</p>
    <section class="summary">
      <strong>诊断结论</strong>
      <p>{escape(failure_category(metrics_dict))}</p>
      <p><strong>Query:</strong> {escape(sample["query"])}</p>
      <p><strong>Retrieved:</strong> <code>{escape(", ".join(sample["retrieved_ids"]))}</code></p>
      <p><strong>Ground Truth:</strong> <code>{escape(", ".join(sorted(sample["relevant_ids"])))}</code></p>
    </section>
    <section class="grid">{"".join(cards)}</section>
    <h2>Metric Details</h2>
    <table>
      <thead><tr><th>指标</th><th>得分</th><th>阈值</th><th>状态</th><th>建议</th></tr></thead>
      <tbody>{"".join(rows)}</tbody>
    </table>
    <h2>Priority Actions</h2>
    <ol>{"".join(priority_items)}</ol>
  </main>
</body>
</html>
"""
    output_path.write_text(html, encoding="utf-8")

---
# Part 5 · 运行诊断流水线

计算四项指标 → 生成诊断 → 打印终端报告 → 输出 `diagnostic_report.html`。

In [ ]:
print("=" * 65)
print("RAG 完整诊断流水线")
print("=" * 65)
print(f"\n📋 查询：{RAG_SAMPLE['query']}")
print(f"📦 检索文档：{RAG_SAMPLE['retrieved_ids']}")
print(f"✅ 相关文档（Ground Truth）：{RAG_SAMPLE['relevant_ids']}\n")
print("正在计算各项指标...\n")

context_recall = compute_recall(RAG_SAMPLE["retrieved_ids"], RAG_SAMPLE["relevant_ids"])
context_precision = compute_precision(RAG_SAMPLE["retrieved_ids"], RAG_SAMPLE["relevant_ids"])

if client:
    print("  [3/4] 调用 DeepSeek 计算 Faithfulness（需要几秒）...")
    faithfulness = _faithfulness_llm(RAG_SAMPLE["answer"], RAG_SAMPLE["context"], client)
else:
    faithfulness = MOCK_FAITHFULNESS
    print(f"  [3/4] Faithfulness 使用 Mock 分数：{faithfulness}")

answer_relevancy = _answer_relevancy_keyword(RAG_SAMPLE["query"], RAG_SAMPLE["answer"])

metrics = {
    "context_recall":    context_recall,
    "context_precision": context_precision,
    "faithfulness":      faithfulness,
    "answer_relevancy":  answer_relevancy,
}

diagnostics = generate_diagnostics(metrics)
failed_items = priority_rank(diagnostics)

print("\n" + "=" * 65)
print("  📊  RAG 诊断报告")
print("=" * 65)
print(f"\n{'指标':<28} {'得分':>6}  {'阈值':>6}  {'状态':>4}")
print("-" * 55)

for d in diagnostics:
    status = "✅ 通过" if d["pass"] else "❌ 未达标"
    bar = render_score_bar(d["score"])
    print(f"{d['metric']:<28} {d['score']:>6.2f}  {d['threshold']:>6.2f}  {status}")
    print(f"  [{bar}]")

print("\n" + "=" * 65)

if not failed_items:
    print("\n🎉 恭喜！所有指标均达标，RAG 系统表现优秀！")
else:
    print(f"\n⚠️  发现 {len(failed_items)} 项指标未达标，以下是优先级建议：\n")
    for i, d in enumerate(failed_items, 1):
        gap = d["threshold"] - d["score"]
        print(f"  【优先级 {i}】{d['metric']}")
        print(f"    当前分数：{d['score']:.2f}  /  阈值：{d['threshold']:.2f}  /  缺口：{gap:.2f}")
        print(f"    建议：{d['recommendation']}")
        print()

print("=" * 65)
print("\n📌 指标改进路径总结：")
print("  Context Recall 低  →  增大 top_k / 减小 chunk_size")
print("  Context Precision 低 →  添加 Reranker / 减小 top_k")
print("  Faithfulness 低   →  优化 System Prompt / 添加引用约束")
print("  Answer Relevancy 低 →  Query Expansion / HyDE / 改写问题")
print("=" * 65)

report_path = Path("diagnostic_report.html")
write_html_report(RAG_SAMPLE, metrics, diagnostics, failed_items, report_path)
print(f"\n已生成 HTML 报告：{report_path.resolve()}")

---
# Chapter 7 · 生产化

> 评估不是一次性任务，是持续工程

## 评估数据集构建

| 阶段 | 样本数 | 用途 |
|------|--------|------|
| MVP 快速验证 | 20-50 条 | 方向是否正确 |
| 迭代优化 | 50-100 条 | 调参和版本对比 |
| 生产验收 | 100-200 条 | 上线前评估 |
| 持续监控 | 200+ 条 | 多维度覆盖 |

**数据集构成建议**：40% 事实类 + 20% 多跳 + 20% 边界 case + 20% 推理类。

## 持续评估三层 + 框架选型

```mermaid
%%{init: {'flowchart': {'curve': 'linear', 'rankSpacing': 45}}}%%
flowchart TB
    L1[Layer1 实时监控] --> L2[Layer2 定期评估]
    L2 --> L3[Layer3 发布门禁]
    L3 --> RAGAS[RAGAS 算分]
    RAGAS --> DIAG[自研诊断层 本 Notebook]
```

| 框架 | 推荐场景 |
|------|----------|
| **RAGAS** | 快速原型、LangChain 生态 |
| **TruLens** | 企业级 trace 监控 |
| **DeepEval** | RAG + Agent 统一评估 |

## RAGAS 告警阈值

- `context_precision` < 0.5：Retriever 严重失调
- `faithfulness` < 0.7：Generator 幻觉严重
- `answer_relevancy` < 0.75：答案跑题

## 踩坑 Top 5

1. **Judge 模型不能省钱**——至少 DeepSeek-V3 或 GPT-4o-mini
2. **GT 只标「有关」没标「必需」**——Recall 虚高
3. **评估集数据泄露**——问题不能照抄 chunk 原文
4. **改 chunk_size 后没重建 GT**——指标波动
5. **P/R 同时优化陷入调参漩涡**——先定 Recall 目标再最大化 Precision

## 课程总结 + 作业

**今天学了什么**

- 5 个 RAG 指标的**数学公式** + **手写 Python 实现**
- **LLM-as-Judge** 的 Faithfulness 验证流程
- **Ground Truth 生成**的两种策略
- **诊断决策树**：指标低了有系统化排查路径

**课后作业（三选一）**

1. **基础版**：对本节 RAG 系统跑完整 5 指标评估，输出诊断报告
2. **进阶版**：构建 50 条评估数据集，对比两种 chunking 策略的 Recall
3. **挑战版**：实现 `RAGDiagnostics` 类，并集成 RAGAS 对比验证手写指标